# DDPG LIME Explanations

In [1]:
# !pip install lime

In [2]:
import os
import numpy as np
import pickle
import random
import pandas as pd
import copy
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

# LIME
import lime
from lime import lime_tabular
from lime.lime_tabular import LimeTabularExplainer

In [3]:
os.environ["CUDA_VISIBLE_DEVICES"]="6"

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [5]:
seed = 316
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [6]:
def convert(seconds):
    seconds = seconds % (24 * 3600)
    hour = seconds // 3600
    seconds %= 3600
    minutes = seconds // 60
    seconds %= 60

    return "%d:%02d:%02d" % (hour, minutes, seconds)

# Load Data

In [7]:
# with open('../data/user_sequences.pkl', 'rb') as f:
#     user_sequences = pickle.load(f)
with open('../data/train_sequences.pkl', 'rb') as f:
    train_sequences = pickle.load(f)
with open('../data/test_sequences.pkl', 'rb') as f:
    test_sequences = pickle.load(f)

# train_sequences and test_sequences are dictionaries like user_sequences
# Keys: UserIDs (ints)
# Values: list of tuples, each tuple is (movieID, rating)

In [8]:
with open('../data/user2idx.pkl', 'rb') as f:
    user2idx = pickle.load(f)
# with open('../data/idx2user.pkl', 'rb') as f:
#     idx2user = pickle.load(f)
# with open('../data/movie2idx.pkl', 'rb') as f:
    # movie2idx = pickle.load(f)
with open('../data/idx2movie.pkl', 'rb') as f:
    idx2movie = pickle.load(f)

In [9]:
# with open('../data/user_sequences_idx.pkl', 'rb') as f:
#     user_sequences_idx = pickle.load(f)

with open('../data/train_sequences_idx.pkl', 'rb') as f:
    train_sequences_idx = pickle.load(f)

with open('../data/test_sequences_idx.pkl', 'rb') as f:
    test_sequences_idx = pickle.load(f)

In [10]:
user_embeddings = np.load('../data/user_embeddings.npy')
movie_embeddings = np.load('../data/movie_embeddings.npy')

print(f"User embeddings: {user_embeddings.shape}") # should be 6040
print(f"Movie embeddings: {movie_embeddings.shape}") # should be 3706

User embeddings: (6040, 100) (should match 6040 users)
Movie embeddings: (3706, 100) (should match 3706 movies)


In [11]:
pmf_raw_data = pd.read_csv("../data/movielens_1M_pmf_raw_data.csv")
print(f"Loaded {len(pmf_raw_data)} ratings")

Loaded 1000209 ratings


In [12]:
# Load the ratings dictionary
with open('../data/ratings_dict.pkl', 'rb') as f:
    ratings_dict = pickle.load(f)

print(f"Loaded {len(ratings_dict)} entries into ratings_dict")

Loaded 1000209 entries into ratings_dict


In [13]:
# type(ratings_dict)
# list(ratings_dict.keys())

In [14]:
# File paths
DATA_DIR = "../data/ml-1m"
RATINGS_FILE = os.path.join(DATA_DIR, "ratings.dat")
USERS_FILE = os.path.join(DATA_DIR, "users.dat")
MOVIES_FILE = os.path.join(DATA_DIR, "movies.dat")

# Column names
ratings_cols = ['user_id', 'movie_id', 'rating', 'timestamp']
users_cols = ['user_id', 'gender', 'age', 'occupation', 'zip_code']
movies_cols = ['movie_id', 'title', 'genres']

# Load data
ratings = pd.read_csv(RATINGS_FILE, sep='::', engine='python', names=ratings_cols, encoding='latin-1')
users = pd.read_csv(USERS_FILE, sep='::', engine='python', names=users_cols, encoding='latin-1')
movies = pd.read_csv(MOVIES_FILE, sep='::', engine='python', names=movies_cols, encoding='latin-1')

In [15]:
# Dictionary that maps MovieID to Title for Quick Lookup
with open('../data/movie_id_to_title.pkl', 'rb') as f:
    movie_id_to_title = pickle.load(f)

# Load Model

In [16]:
class Actor(nn.Module):
    """
    Randomly initialize the Actor πθ
    Thus assuming standard implementation.
    
    maps: state -> action
    From paper: "By two ReLU layers and one Tanh layer, the state representation s 
    is transformed into an action a = π_θ(s)"
    """
    def __init__(self, state_dim, action_dim=100, hidden_dim=256):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, action_dim)
        
        # Initialize weights
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.xavier_uniform_(self.fc3.weight)
        
    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        # Tanh maps action to [-1, 1]
        action = torch.tanh(self.fc3(x))
        return action

In [17]:
# Load Actor and Critic models
version_number = 2

state_dim = 300
actor = Actor(state_dim)
actor = actor.to(device)
# Load the weights into that architecture
actor.load_state_dict(torch.load(f'../models/drr_actor_weights_{version_number}.pth'))
actor.eval()

/tmp/ipykernel_4057057/187023785.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  actor.load_state_dict(torch.load(f'../models/drr_actor_weights_{version_number}.pth'))


Actor(
  (fc1): Linear(in_features=300, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=256, bias=True)
  (fc3): Linear(in_features=256, out_features=100, bias=True)
)

# Create LIME functions

In [18]:
# a Wrapper that perturbs the HISTORY items
class HistoryExplainerWrapper:
    def __init__(self, actor, user_idx, history_indices, movie_embeddings, user_embeddings, target_item_idx, device):
        self.actor = actor
        self.u = user_embeddings[user_idx]
        self.history_indices = history_indices
        self.movie_embeddings = movie_embeddings
        self.target_emb = torch.FloatTensor(movie_embeddings[target_item_idx]).to(device)
        self.device = device

    def predict(self, masks):
        """
        masks: (N, 5) binary array. 1 means "keep movie in history", 0 means "remove it".
        """
        scores = []
        for mask in masks:
            # print(f"MASK: {mask}")
            # Filter history based on mask (if all 0, use a dummy)
            active_indices = [self.history_indices[i] for i in range(len(mask)) if mask[i] > 0.5]
            if not active_indices: active_indices = [self.history_indices[0]] 
            
            # Recompute 'g' (the DRR-ave history average)
            g = np.mean(self.movie_embeddings[active_indices], axis=0)
            u_g = self.u * g
            state = np.concatenate([self.u, u_g, g])
            
            # Get Actor Score
            with torch.no_grad():
                state_t = torch.FloatTensor(state).to(self.device).unsqueeze(0)
                action = self.actor(state_t).squeeze(0)
                score = torch.dot(action, self.target_emb).item()
            scores.append(score)
        return np.array(scores)

In [19]:
def clean_movie_title(text):
    # 1. Split by '=' and take the first part
    step1 = text.split('=')[0]
    # 2. Split that result by '>' and take the first part
    step2 = step1.split('>')[0]
    
    final_title = step2.strip()
    return final_title

In [20]:
def get_influence_label(weight):
    # Check if the weight is a positive number
    if weight > 0:
        return "+"
    else:
        # This handles both negative numbers AND exactly 0
        return "-"

In [21]:
# Load the saved trajectory
with open('../results/baseline_session_trajectories.pkl', 'rb') as f:
    sessions = pickle.load(f)

In [22]:
users_to_explain = [1, 3, 40, 67, 316]

for user_id in users_to_explain:
    session = sessions[user_id]
    print(f"\n{'='*60}")
    print(f"USER {user_id}")
    for step in session['steps']:
        print(f"{step['step']}: {step['recommended_movie_title']} (rating {step['rating']})")

    for step in session['steps']:
        print(f"\nStep {step['step']}: Recommended {step['recommended_movie_title']} (rating {step['rating']})")
        # Get history at this moment
        history_at_time = step['history_at_time']
        history_titles = [movie_id_to_title[idx2movie[idx]] for idx in history_at_time]
        history_ratings = [ratings_dict.get((user2idx[user_id], idx), 'N/A') for idx in history_at_time]


        # Create LIME explainer
        background = np.random.randint(0, 2, size=(500, len(history_at_time)))
        explainer = LimeTabularExplainer(
            training_data=background,
            feature_names=history_titles,
            categorical_features=list(range(len(history_at_time))),
            mode='regression'
        )
        
        # Create wrapper
        wrapper = HistoryExplainerWrapper(
            actor, 
            user2idx[user_id], 
            history_at_time,
            movie_embeddings, 
            user_embeddings, 
            step['recommended_movie_idx'], 
            device
        )
        
        # Explain
        exp = explainer.explain_instance(
            np.ones(len(history_at_time)),
            wrapper.predict,
            num_features=len(history_at_time)
        )
        
        # Print results
        exp_list = exp.as_list()
        
        # Create title-to-rating mapping BEFORE creating DataFrame
        title_to_rating = {}
        for title, idx in zip(history_titles, history_at_time):
            cleaned = clean_movie_title(title)
            rating = ratings_dict.get((user2idx[user_id], idx), 'N/A')
            title_to_rating[cleaned] = rating
        
        # Now create DataFrame with correct ratings
        df = pd.DataFrame(exp_list, columns=["Movie in History", "Weight"])
        df["Movie in History"] = df["Movie in History"].apply(clean_movie_title)
        df["Rating"] = df["Movie in History"].map(title_to_rating)  # Map using the dictionary
        df["Influence"] = df["Weight"].apply(get_influence_label)
        df["Weight"] = df["Weight"].round(2)
        df = df.sort_values("Weight", ascending=False)
                
        print(df[["Movie in History", "Rating", "Weight", "Influence"]].to_string())


USER 1
1: Tarzan (1999) (rating 3)
2: Hercules (1997) (rating 4)
3: Toy Story (1995) (rating 5)
4: Pocahontas (1995) (rating 5)
5: Beauty and the Beast (1991) (rating 5)
6: Hunchback of Notre Dame, The (1996) (rating 4)
7: Bug's Life, A (1998) (rating 5)
8: Mulan (1998) (rating 4)
9: Close Shave, A (1995) (rating 3)
10: Aladdin (1992) (rating 4)

Step 1: Recommended Tarzan (1999) (rating 3)
                         Movie in History  Rating  Weight Influence
0                          Ponette (1996)     4.0    1.58         +
4  Snow White and the Seven Dwarfs (1937)     4.0   -0.09         -
3                 Schindler's List (1993)     5.0   -0.19         -
2                            Dumbo (1941)     5.0   -0.47         -
1           Miracle on 34th Street (1947)     4.0   -0.82         -

Step 2: Recommended Hercules (1997) (rating 4)
                         Movie in History  Rating  Weight Influence
0                          Ponette (1996)     4.0    0.93         +
3            